# Module 4 Lab 2: First boto3 Calls and Token Cost Math

**20 points · estimated time 75 to 105 minutes**

> **The one sentence version:** call a foundation model from Python, read the token counts out of the response, and work out what your feature would cost at real volume.

**Student name:** `SHANE MCQUEEN`
**Date run:** `11/11/2026`
**GitHub portfolio repository URL:** `https://github.com/ShaneMac13/aws-ai-portfolio`. 

### What you will understand when you finish
- Create a boto3 client and confirm which identity and Region it is using
- Tell the Bedrock **control plane** (`bedrock`) apart from the **runtime plane** (`bedrock-runtime`) and know which one to call
- Read input and output token counts out of a real model response
- Compute the monthly cost of a feature from measured tokens and published rates
- Explain why output tokens are the expensive half, and prove it with your own numbers

### Lab steps
1. Check your setup (identity, Region, boto3 version)
2. List the models you have, then make one real Converse call
3. Ask the same thing two ways and compare the tokens
4. Turn your numbers into a cost

Then: Reflection (your own words, **no AI assistance**), troubleshooting, and the submission checklist.

## Step 1: Check your setup *(about 10 minutes, 2 points)*

Print three things before you do anything else: **which identity** your code is running as, **which Region** it will call, and your **boto3 version**. Do not assume any of them.

**You should see:** your IAM user ARN, a Region name, and two version numbers.

In [7]:
#Clear env variables
import os
os.environ.pop('AWS_DEFAULT_PROFILE', None)
os.environ.pop('AWS_PROFILE', None)


import boto3, botocore

session = boto3.Session(profile_name='smcqueen-lab')
control = session.client('bedrock')

sts = session.client('sts')
me = sts.get_caller_identity()

print(me["Account"])
print(me["Arn"])
print(boto3.session.Session().region_name)


# Versions: boto3 and botocore must be recent enough to know the Converse operation.
print(boto3.__version__, botocore.__version__)

647253403357
arn:aws:iam::647253403357:user/smcqueen-lab
us-east-1
1.43.118 1.43.118


## Step 2: See what models you have, then call one *(about 20 minutes, 5 points)*

Use the **control plane** client to list the models in your Region and print how many came back. Then switch to the **runtime** client and make one Converse call with a short question. Print the whole response once, then print just the usage block and just the text.

**You should see:** a count of models, ten model ids, and then a usage block with three token counts and a text answer.

### 2a. List the models (control plane)

In [11]:
# Control plane client: asks ABOUT models (what exists in this Region). It does not run them.
control = session.client('bedrock')
models = control.list_foundation_models()['modelSummaries']
print(len(models), 'models listed in this Region')

# Show the first ten model ids and providers. Pick your model id from this list (or the full list).
for m in models[:99]:
    print(m['modelId'], m['providerName'])

99 models listed in this Region
openai.gpt-6-astra OpenAI
nvidia.nemotron-nano-12b-v2 NVIDIA
openai.gpt-5.6-terra OpenAI
anthropic.claude-haiku-4-5-20251001-v1:0 Anthropic
qwen.qwen3-235b-a22b-2507-v1:0 Qwen
anthropic.claude-opus-5-5 Anthropic
moonshotai.kimi-k2.5 Moonshot AI
openai.gpt-oss-120b-1:0 OpenAI
stability.stable-creative-upscale-v1:0 Stability AI
zai.glm-5.3 Z.AI
qwen.qwen3-next-80b-a3b Qwen
moonshotai.kimi-k3 Moonshot AI
anthropic.claude-fable-5 Anthropic
deepseek.v3.2 DeepSeek
anthropic.claude-haiku-5-5 Anthropic
twelvelabs.pegasus-1-5-v1:0 TwelveLabs
nvidia.nemotron-nano-3-30b NVIDIA
anthropic.claude-sonnet-4-6 Anthropic
minimax.minimax-m2 MiniMax
zai.glm-4.7-flash Z.AI
xai.grok-4.7 xAI
mistral.voxtral-mini-3b-2507 Mistral AI
xai.grok-4.6 xAI
amazon.nova-pro-v1:0 Amazon
stability.stable-image-remove-background-v1:0 Stability AI
stability.stable-image-control-sketch-v1:0 Stability AI
amazon.nova-2-lite-v1:0 Amazon
stability.stable-conservative-upscale-v1:0 Stability AI
min

### 2b. Make one real Converse call (runtime plane)

> **Hint:** listing a model does not mean you can call it. If the call returns `AccessDeniedException`, go to **Model access** in the Bedrock console, request that model, and try again. That is the most common reason a first Bedrock call fails, and it is not a bug in your code.

In [14]:
MODEL_ID = 'deepseek.v3.2'

QUESTION = 'what does chicken taste like?'

assert MODEL_ID, "Fill in MODEL_ID above (TODO (human input))."
assert QUESTION, "Fill in QUESTION above (TODO (human input))."

# Runtime client: this one actually TALKS to a model. converse() lives here, not on 'bedrock'.
runtime = session.client('bedrock-runtime')
resp = runtime.converse(
    modelId=MODEL_ID,
    # content is a LIST of dictionaries, each with a 'text' key. Keep this shape exactly.
    messages=[{'role': 'user',
               'content': [{'text': QUESTION}]}],
    inferenceConfig={'maxTokens': 200},   # cap on output tokens for this call (value from the lab)
)

print(resp)                                          # the whole thing, once (required by the checklist)
print(resp['usage'])                                 # the bill: inputTokens, outputTokens, totalTokens
print(resp['output']['message']['content'][0]['text'])   # just the answer text

{'ResponseMetadata': {'RequestId': 'b985f2b0-2d87-40cc-bee3-1ad983ed0479', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Wed, 11 Nov 2026 23:42:18 GMT', 'content-type': 'application/json', 'content-length': '999', 'connection': 'keep-alive', 'x-amzn-requestid': 'b985f2b0-2d87-40cc-bee3-1ad983ed0479'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'text': 'Describing the taste of chicken is surprisingly tricky because it\'s often seen as a "blank canvas" for flavors, but it has its own distinct character. Here’s a breakdown:\n\n### The Basic Taste of Plain, Cooked Chicken (e.g., roasted or poached):\n*   **Mild & Savory:** Its primary flavor is mild, savory, and slightly sweet. It\'s not bold or gamey like red meats.\n*   **Umami:** It has a subtle umami quality (the savory, meaty "fifth taste").\n*   **Texture is Key:** The taste experience is heavily influenced by texture. **White meat** (breast) is lean, tender, and can be mild to the point of being 

> **Checkpoint:** you have a real response with real token counts from your own account.
> **Do not continue until this is true.**

## Step 3: Ask the same thing two ways and compare the tokens *(about 25 minutes, 5 points)*

Pick one small task, such as sorting a support message into a department. Write two prompts for it:
- **prose**: asks for a paragraph.
- **structured**: asks for only a short structured answer, such as a JSON object with two fields.

Send the **same input** through both and write down the input and output token counts for each.

> **Hint:** keep the input as close to identical as you can. If you change the input text as well as the instruction, you won't know which change moved the numbers.

> **Notebook layout note:** the lab's template has two blank prompt strings. To keep the input identical, this notebook splits each prompt into a **shared input text** plus a **per-design instruction**, and joins them. If you'd rather type two full prompts as in the lab, replace `prose` and `structured` directly. The input text and both instructions are yours to write.

In [23]:
# TODO (human input): the ONE piece of input both prompts will process, e.g. a support message you
# write yourself. Use the same text for both designs. Don't reuse the lab's sample ticket or numbers.
INPUT_TEXT = "Ticket 2112 is a technical question. The customer needs instruction on email. Route to product support, priority normal."

PROSE_INSTRUCTION = 'write an email to product support to check this ticket'

STRUCTURED_INSTRUCTION = 'write an email to product support to check this ticket. respond only: subject [TICKET_NUMBER] - [PRIORITY], date mm/dd/yyyy, time 12hr 00:00A/P, Priority, Department, Ticket Number'

assert INPUT_TEXT and PROSE_INSTRUCTION and STRUCTURED_INSTRUCTION, \
    "Fill in INPUT_TEXT, PROSE_INSTRUCTION and STRUCTURED_INSTRUCTION (TODO (human input))."

# Same input, different instruction: only the requested SHAPE of the answer changes.
# (The order of instruction and input text here is a choice, not something the lab specifies.)
prose      = f"{PROSE_INSTRUCTION}\n\n{INPUT_TEXT}"
structured = f"{STRUCTURED_INSTRUCTION}\n\n{INPUT_TEXT}"

results = {}   # keep each usage block so the ratio and Step 4 can use the measured numbers
for name, prompt in [('prose', prose), ('structured', structured)]:
    r = runtime.converse(
        modelId=MODEL_ID,                     # same model as Step 2
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 200},   # same cap for both designs (value from the lab)
    )
    print(name, r['usage'])
    # Also show the answer so you can check both designs actually did the same job.
    print('   text:', r['output']['message']['content'][0]['text'])
    results[name] = r['usage']

prose {'inputTokens': 39, 'outputTokens': 170, 'totalTokens': 209}
   text: Of course. Here is a professional and clear email you can use.

---

**Subject:** Inquiry Regarding Ticket #2112: Routing to Product Support

**Dear Product Support Team,**

I am writing to bring Ticket #2112 to your attention for further handling.

The customer's inquiry is a technical question, specifically requiring instruction on an email-related issue. As this falls within your team's expertise, please take over this ticket.

**Ticket Details:**
*   **Ticket Number:** #2112
*   **Issue Type:** Technical Support / How-to Instructions
*   **Topic:** Email functionality
*   **Priority:** Normal

The ticket is now ready in the queue for your review. Please proceed with assisting the customer at your earliest convenience.

Thank you,

**[Your Name]**
[Your Title/Team, if applicable]
structured {'inputTokens': 82, 'outputTokens': 46, 'totalTokens': 128}
   text: Subject: TICKET_2112 - NORMAL  

Date: 05/15/2023 

In [24]:
# Ratio of output tokens: how many times more output the prose design produced.
prose_out      = results['prose']['outputTokens']
structured_out = results['structured']['outputTokens']
print('prose      input', results['prose']['inputTokens'],      ' output', prose_out)
print('structured input', results['structured']['inputTokens'], ' output', structured_out)
print(f'output token ratio (prose / structured) = {prose_out / structured_out:.1f}x')

prose      input 39  output 170
structured input 82  output 46
output token ratio (prose / structured) = 3.7x


> **Checkpoint:** both prompts do the same job. They differ in the shape of the answer you asked for, not in the task.
> **Do not continue until this is true.**

## Step 4: Turn your numbers into a cost *(about 20 minutes, 5 points)*

Invent a plausible feature for a real or imagined business. Use the token counts you measured for the **structured** prompt, because that's the design you would actually ship. Look up the current published rate for your model, compute the cost of one call, then multiply up to 1,000, 50,000 and 2,000,000 calls a month. Cite the pricing page and the date you read it. Finish with one sentence saying whether the biggest figure is plausible for that business.

**Feature (one line, your invention):** `Ticket Routing`

**Pricing source:** `https://aws.amazon.com/bedrock/pricing/` 
**Date you read it:** `11/11/2026`

> **Hint:** rates are published per **thousand** tokens or per **million** tokens depending on the page. Check which one you're reading. Being out by a thousand is the easiest mistake in this lab to make and the hardest to spot. The formula below expects dollars **per 1,000 tokens**, so divide a per-million rate by 1,000.

**You should see:** a cost per call, three monthly figures, a cited pricing page with the date, and your sanity check sentence.

In [25]:
# Measured token counts from Step 3, structured design (the one you would ship).
# Pulled from the results above so they are YOUR numbers, not the lab's example (385 / 18).
in_tokens  = results['structured']['inputTokens']    # your measured number
out_tokens = results['structured']['outputTokens']   # your measured number

# per 1,000,000 tokens, divide it by 1,000 before entering it here.
in_rate  = 0.58 / 1000    # dollars per 1,000 input tokens

out_rate = 1.68 / 1000    # dollars per 1,000 output tokens

assert in_rate is not None and out_rate is not None, \
    "Fill in in_rate and out_rate in dollars per 1,000 tokens (TODO (human input))."

# Cost of one call: each token count / 1000 times its per-1,000 rate.
per_call = (in_tokens / 1000 * in_rate) + (out_tokens / 1000 * out_rate)

# Arithmetic shown explicitly (the rubric asks for the arithmetic to be visible).
print(f'cost per call = ({in_tokens} / 1000 x {in_rate}) + ({out_tokens} / 1000 x {out_rate})')
print('per call', per_call)
print(f'per call ${per_call:.8f}')

# Scale to monthly volumes given in the lab.
for volume in [1000, 50000, 2000000]:
    print(volume, 'calls a month costs', volume * per_call, f'(${volume * per_call:,.2f})')

# Optional extra check: how much of the per-call cost is output vs input.
print(f'input share  ${in_tokens / 1000 * in_rate:.8f}')
print(f'output share ${out_tokens / 1000 * out_rate:.8f}')

cost per call = (82 / 1000 x 0.00058) + (46 / 1000 x 0.0016799999999999999)
per call 0.00012484
per call $0.00012484
1000 calls a month costs 0.12484 ($0.12)
50000 calls a month costs 6.242 ($6.24)
2000000 calls a month costs 249.68 ($249.68)
input share  $0.00004756
output share $0.00007728


**Sanity check sentence** (is the 2,000,000-calls-a-month figure plausible for that business?):

## Troubleshooting (from the lab)

| Symptom | Likely cause | What to do |
|---|---|---|
| `AccessDeniedException` on `converse` | Model access not granted for that model in that Region | Bedrock console → Model access → request the model, try again. Approval is usually immediate. |
| The operation does not exist | Wrong client | `list_foundation_models` is on `bedrock`; `converse` is on `bedrock-runtime`. |
| `ValidationException` on the request | The `messages` structure isn't quite right | `content` is a list of dictionaries, each with a `text` key. Copy the shape exactly, then change the words. |
| No usage block in the response | Old boto3 or botocore | `%pip install --upgrade boto3 botocore`, restart the kernel, run again. |
| Cost comes out absurdly large or small | Rate read per million but used per thousand, or the reverse | Check the units on the pricing page. |
| `ThrottlingException` | Too many calls too quickly | Wait a few seconds and run the cell again. |